# Part 2: Commit time series and gaps

In [ ]:
NETID = "sydlwils"
OUT = "."

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, glob
df = pd.read_csv("sydlwils_project_summary.csv", sep=";", dtype=str, keep_default_na=False)
df = df.drop_duplicates(["project","commit"])  # every commit appeared twice in Part 1 output
df["ts"] = pd.to_datetime(pd.to_numeric(df["time"]), unit="s", utc=True)
df["month"] = df["ts"].dt.strftime("%Y-%m")
df.groupby("project").size()

In [ ]:
rows = []
series = {}
for proj, g in df.groupby("project"):
    counts = g.groupby("month").size()
    idx = pd.period_range(counts.index.min(), counts.index.max(), freq="M")
    ts = counts.reindex(idx.strftime("%Y-%m"), fill_value=0)
    series[proj] = ts
    out = pd.DataFrame({"Month": ts.index, "#Commits": ts.values})
    out.to_csv(f"{NETID}_commits_timeseries_{proj}.csv", sep=";", index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(out["Month"], out["#Commits"], marker="o", linestyle="-", markersize=3)
    plt.xlabel("Time (YYYY-MM)")
    plt.ylabel("Number of Commits")
    plt.title(proj)
    plt.grid(True)
    n = len(out)
    step = max(1, n // 24)
    plt.xticks(range(0, n, step), out["Month"][::step], rotation=45)
    plt.tight_layout()
    plt.savefig(f"{NETID}_timeseries_{proj}.png", dpi=300)
    plt.close()

    # gaps: runs of consecutive zero months
    runs = []
    start = None
    vals = ts.values
    for i, v in enumerate(vals):
        if v == 0 and start is None:
            start = i
        if v != 0 and start is not None:
            runs.append((start, i - 1))
            start = None
    # series begins and ends on a nonzero month so no trailing run
    if runs:
        s, e = max(runs, key=lambda r: (r[1] - r[0], -r[0]))  # earliest of ties
        length = e - s + 1
        after = int(vals[e + 1:].sum())
        row_gap = (ts.index[s], ts.index[e], length, after)
    else:
        row_gap = ("", "", 0, int(vals.sum()))
    ngaps3 = sum(1 for s_, e_ in runs if e_ - s_ + 1 >= 3)

    rows.append({
        "Project": proj,
        "ncommits": len(g),
        "nauthors": g["author"].nunique(),
        "from": g["ts"].min().strftime("%Y-%m-%d %H:%M:%S"),
        "to": g["ts"].max().strftime("%Y-%m-%d %H:%M:%S"),
        "nstars": "", "nforks": "", "lastGHCommitDate": "",
        "LongestGapStart": row_gap[0], "LongestGapEnd": row_gap[1],
        "LongestGapLength": row_gap[2], "NumCommitsAfterLastGap": row_gap[3],
        "ActivityPattern": "",
        "NumberOfGapsInTimeline": ngaps3,
        "nmonths": n, "zero_months": int((vals == 0).sum()),
    })



In [ ]:
stats = pd.DataFrame(rows)
stats

## Interpretation

- **google_uis-rnn**: Declining. Burst of activity at the start (2018-10 to 2019), then near-zero. 3 gaps of 3+ months; the longest is 23 months (2021-08 to 2023-06).
- **inab_wes-backend**: Cyclical. Repeating bursts and lulls roughly every 8-12 months. 0 gaps of 3+ months.
- **knossos-project_knossos**: Declining. Peak around 2014-2016, then a slow decline to a handful of commits per month. 2 gaps of 3+ months.
- **leftygray_cascade_calculations**: Declining. Most activity in 2015-2019, then sparse with occasional spikes. 3 gaps of 3+ months.
- **nrel_openstudio-server**: Declining. Huge early spike (2013), then gradually tapering. 1 gap of 3+ months.
- **openmd_openmd**: Steady. After an initial burst in 2004, a low, consistent level of activity for two decades. 1 gap of 3+ months.
- **parflow_parflow**: Rising. Low activity until about 2016, then much higher and growing through 2026. 6 gaps of 3+ months, mostly early.
- **project-konda_pattern-based-quality-analysis**: Cyclical. Repeated large peaks and troughs, roughly one per year. 0 gaps of 3+ months.
- **wistaria_materiappsinstaller**: Irregular. Sporadic spikes (2017, 2019-2020) without a clear trend. 3 gaps of 3+ months.
- **wtmatlock_flanker**: Declining. One large burst in 2020-2021, then almost no activity. 6 gaps of 3+ months; longest is 9 months (2022-03 to 2022-11).

Note: patterns are judged by eye from the plots; gap counts count runs of 3 or more consecutive zero-commit months.